# Cluster Czkawka groups

`_groups.json` is loaded into a pandas dataframe. Pick a source-group range and which columns UMAP should see, then cluster with HDBSCAN. Click a point to show that cluster's images. The last cell writes `clusters_groups.json`.


In [ ]:
from io import BytesIO
from pathlib import Path
import sys

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from PIL import Image

ROOT = Path.cwd().resolve()
if not (ROOT / "photo_repair").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from photo_repair.embedding import (
    HASH_COLUMNS,
    embed_and_cluster,
    load_dataframe,
    select_groups,
    summarize,
    to_czkawka_groups,
    write_groups,
)

images = load_dataframe(ROOT / "_groups.json")
print(f"{len(images)} images in {int(images['source_group'].nunique())} source groups")
images[["source_group", "path", "width", "height", "size", "difference"] + HASH_COLUMNS[:]].head()


## Parameters

Group numbers are 1-based and the range is inclusive. Hash bytes are one toggle: on means all 32 `hash_00`…`hash_31` columns. The other boxes are single columns.


In [ ]:
group_start = widgets.BoundedIntText(value=1, min=1, max=int(images["source_group"].max()), description="group start")
group_end = widgets.BoundedIntText(value=int(images["source_group"].max()), min=1, max=int(images["source_group"].max()), description="group end")
use_hash = widgets.Checkbox(value=True, description="hash bytes")
use_source_group = widgets.Checkbox(value=True, description="source_group")
use_size = widgets.Checkbox(value=False, description="size")
use_width = widgets.Checkbox(value=False, description="width")
use_height = widgets.Checkbox(value=False, description="height")
use_modified = widgets.Checkbox(value=False, description="modified_date")
use_difference = widgets.Checkbox(value=False, description="difference")
n_neighbors = widgets.IntText(value=2, description="neighbors")
min_dist = widgets.FloatText(value=0.0, description="min_dist")
metric = widgets.Dropdown(options=["euclidean", "manhattan", "cosine", "chebyshev", "hamming"], value="euclidean", description="metric")
n_components = widgets.ToggleButtons(options=[2, 3], value=2, description="dimensions")
seed = widgets.IntText(value=None, description="seed")
min_cluster_size = widgets.IntText(value=2, description="min cluster")
min_samples = widgets.IntText(value=2, description="min samples")

def selected_columns():
    names = []
    if use_hash.value:
        names.extend(HASH_COLUMNS)
    for box, name in (
        (use_source_group, "source_group"),
        (use_size, "size"),
        (use_width, "width"),
        (use_height, "height"),
        (use_modified, "modified_date"),
        (use_difference, "difference"),
    ):
        if box.value:
            names.append(name)
    return names

display(widgets.VBox([
    widgets.HBox([group_start, group_end]),
    widgets.HBox([use_hash, use_source_group, use_size, use_width, use_height, use_modified, use_difference]),
    widgets.HBox([n_neighbors, min_dist, metric, n_components]),
    widgets.HBox([seed, min_cluster_size, min_samples]),
]))


## Embed and cluster

Run this cell to build `result`. Defaults match the run that wrote `clusters_groups.json`.


In [ ]:
chosen = select_groups(images, int(group_start.value), int(group_end.value))
result = embed_and_cluster(
    chosen,
    selected_columns(),
    n_neighbors=int(n_neighbors.value),
    min_dist=float(min_dist.value),
    metric=metric.value,
    n_components=int(n_components.value),
    seed=int(seed.value),
    min_cluster_size=int(min_cluster_size.value),
    min_samples=int(min_samples.value),
)
summarize(result)


## View clusters

Click a point. The cluster number and its images appear underneath the plot.


In [ ]:
image_out = widgets.Output()

def capture_times(paths):
    import json
    import subprocess
    present = [path for path in paths if Path(path).is_file()]
    if not present:
        return {}
    completed = subprocess.run(
        ["exiftool", "-json", "-m", "-DateTimeOriginal", "-CreateDate", "-d", "%Y:%m:%d %H:%M:%S", *present],
        capture_output=True,
        text=True,
    )
    try:
        payload = json.loads(completed.stdout or "[]")
    except json.JSONDecodeError:
        payload = []
    found = {}
    for record in payload:
        source = record.get("SourceFile")
        when = record.get("DateTimeOriginal") or record.get("CreateDate")
        if source:
            found[str(source)] = when or "no capture time"
    return found

def thumbnail(path, when):
    file = Path(path)
    caption = f"<small>{file.name}<br>{when}</small>"
    if not file.is_file():
        return widgets.HTML(f"<p>missing<br>{file.name}</p>")
    with Image.open(file) as picture:
        picture = picture.convert("RGB")
        picture.thumbnail((180, 180))
        buffer = BytesIO()
        picture.save(buffer, format="PNG")
    return widgets.VBox([
        widgets.Image(value=buffer.getvalue(), format="png"),
        widgets.HTML(caption),
    ])

def show_cluster(cluster_id):
    members = result.loc[result["cluster"] == int(cluster_id), "path"].tolist()
    times = capture_times(members)
    image_out.clear_output(wait=True)
    with image_out:
        print(f"cluster {int(cluster_id)}    {len(members)} images")
        display(widgets.GridBox(
            [thumbnail(path, times.get(path, "no capture time")) for path in members],
            layout=widgets.Layout(grid_template_columns="repeat(auto-fill, minmax(190px, 1fr))"),
        ))

def on_click(trace, points, state):
    if not points.point_inds:
        return
    show_cluster(trace.customdata[points.point_inds[0]][0])

customdata = result["cluster"].to_numpy().reshape(-1, 1)
marker = dict(size=4, color=result["cluster"] % 20, colorscale="Turbo")
if int(n_components.value) == 3:
    trace = go.Scatter3d(x=result["x"], y=result["y"], z=result["z"], mode="markers", marker=marker, customdata=customdata)
else:
    trace = go.Scatter(x=result["x"], y=result["y"], mode="markers", marker=marker, customdata=customdata)
figure = go.FigureWidget(data=[trace])
figure.update_layout(height=680, title=f"{result['cluster'].nunique()} clusters")
figure.data[0].on_click(on_click)
display(figure, image_out)


## Write groups JSON

Run this cell to overwrite `clusters_groups.json` with the clusters currently in `result`. One-image clusters are included.


In [ ]:
output_path = ROOT / "clusters_groups.json"
write_groups(output_path, to_czkawka_groups(result))
print(f"wrote {output_path.resolve()}")
summarize(result)
